In [1]:
# Cell 1: Import Libraries
# Purpose: Load all libraries needed for cleaning, analysis and visualization

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully!")
print(f"Pandas version    : {pd.__version__}")
print(f"Numpy version     : {np.__version__}")


All libraries loaded successfully!
Pandas version    : 2.2.2
Numpy version     : 2.0.2


In [2]:
from google.colab import files

In [3]:
uploaded = files.upload()

Saving scored_transactions.csv to scored_transactions.csv


In [4]:
import io

df = pd.read_csv(io.BytesIO(list(uploaded.values())[0]))

print(f"Rows    : {df.shape[0]:}")
print(f"Columns : {df.shape[1]}")
df.head(3)

Rows    : 3199
Columns : 17


,transaction_id,merchant_id,user_id,card_number,transaction_date,transaction_amount,device_id,has_cbk,is_fraud,missing_device,hour_of_day,flag_high_amount,flag_night,flag_multi_card,flag_shared_card,risk_score,risk_label
0,21321153,36617,79054,406655******4572,2019-11-29 00:04:17,2971.56,101848.0,True,1,0,0,1,1,1,1,4,High
1,21321132,1308,96025,406655******5763,2019-11-29 02:02:31,2904.60,438940.0,True,1,0,2,1,1,1,1,4,High
2,21322111,75917,17929,606282******5292,2019-11-22 22:43:09,4095.82,960729.0,True,1,0,22,1,1,1,0,3,High


In [5]:
# Fix 1: Make a clean copy and parse dates
df_clean = df.copy()

df_clean['transaction_date'] = pd.to_datetime(df_clean['transaction_date'], errors='coerce')

print(f"Date type : {df_clean['transaction_date'].dtype}")
print(f"Sample    : {df_clean['transaction_date'][0]}")

Date type : datetime64[ns]
Sample    : 2019-11-29 00:04:17


In [6]:
# Fix 2: Standardize has_cbk to uppercase
df_clean['has_cbk'] = df_clean['has_cbk'].astype(str).str.strip().str.upper()

# Verify
print(df_clean['has_cbk'].value_counts())

has_cbk
FALSE    2808
TRUE      391
Name: count, dtype: int64


In [7]:
# Fix 3: Convert is_fraud to boolean
df_clean['is_fraud'] = df_clean['is_fraud'].astype(bool)

# Verify
print(df_clean['is_fraud'].value_counts())
print(f"Data type : {df_clean['is_fraud'].dtype}")

is_fraud
False    2808
True      391
Name: count, dtype: int64
Data type : bool


In [8]:
# Fix 4: Keep device_id nulls but update missing_device flag
df_clean['missing_device'] = df_clean['device_id'].isna().astype(int)

# Verify
print(df_clean['missing_device'].value_counts())
print(f"Missing devices : {df_clean['missing_device'].sum()}")

missing_device
0    2369
1     830
Name: count, dtype: int64
Missing devices : 830


In [9]:
# Fix 5: Extract time features from transaction_date
df_clean['hour']    = df_clean['transaction_date'].dt.hour
df_clean['date']    = df_clean['transaction_date'].dt.date
df_clean['weekday'] = df_clean['transaction_date'].dt.day_name()
df_clean['month']   = df_clean['transaction_date'].dt.month_name()

# Verify
print(df_clean[['transaction_date','hour','date',
                'weekday','month']].head(3))

     transaction_date  hour        date weekday     month
0 2019-11-29 00:04:17     0  2019-11-29  Friday  November
1 2019-11-29 02:02:31     2  2019-11-29  Friday  November
2 2019-11-22 22:43:09    22  2019-11-22  Friday  November


In [10]:
# Fix 6: Remove duplicate transaction_ids
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset='transaction_id', keep='first')
after = len(df_clean)

# Verify
print(f"Rows before : {before:}")
print(f"Rows after  : {after:}")
print(f"Duplicates removed : {before - after}")

Rows before : 3199
Rows after  : 3199
Duplicates removed : 0


In [11]:
# Fix 7: Set correct data types for numeric columns
df_clean['transaction_amount'] = df_clean['transaction_amount'].astype(float)
df_clean['user_id']            = df_clean['user_id'].astype(int)
df_clean['merchant_id']        = df_clean['merchant_id'].astype(int)
df_clean['transaction_id']     = df_clean['transaction_id'].astype(int)

# Verify
print(df_clean[['transaction_id','user_id',
                'merchant_id',
                'transaction_amount']].dtypes)

transaction_id          int64
user_id                 int64
merchant_id             int64
transaction_amount    float64
dtype: object


In [12]:
# Fix 8: Set correct order for risk_label (Low → Medium → High)
df_clean['risk_label'] = pd.Categorical(
    df_clean['risk_label'],
    categories=['Low', 'Medium', 'High'],
    ordered=True)

# Verify
print(df_clean['risk_label'].value_counts().sort_index())
print(f"\nData type : {df_clean['risk_label'].dtype}")

risk_label
Low       3057
Medium     122
High        20
Name: count, dtype: int64

Data type : category


In [13]:
# Final Validation: Confirm all cleaning is complete
print("=" * 45)
print("CLEANING VALIDATION REPORT")
print("=" * 45)
print(f"Total rows         : {len(df_clean):}")
print(f"Total columns      : {df_clean.shape[1]}")
print(f"\nFraud rows         : {df_clean['is_fraud'].sum():}")
print(f"Clean rows         : {(~df_clean['is_fraud']).sum():}")
print(f"CBK rate           : {df_clean['is_fraud'].mean()*100:.2f}%")
print(f"\nMissing device     : {df_clean['missing_device'].sum():}")
print(f"\nRisk label counts:")
print(df_clean['risk_label'].value_counts().sort_index())
print(f"\nRemaining nulls:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])
print("\n✓ Data is clean and ready for analysis!")

CLEANING VALIDATION REPORT
Total rows         : 3199
Total columns      : 21

Fraud rows         : 391
Clean rows         : 2808
CBK rate           : 12.22%

Missing device     : 830

Risk label counts:
risk_label
Low       3057
Medium     122
High        20
Name: count, dtype: int64

Remaining nulls:
device_id    830
dtype: int64

✓ Data is clean and ready for analysis!


In [14]:
# Chart 1: Overall Transaction Distribution
# Purpose: Show fraud vs clean transaction split

labels = ['Clean', 'Fraud']
values = [df_clean['is_fraud'].value_counts()[False],
          df_clean['is_fraud'].value_counts()[True]]

fig = px.pie(
    names=labels,
    values=values,
    hole=0.6,
    color=labels,
    color_discrete_map={
        'Clean' : '#00B894',
        'Fraud' : '#D63031'
    },
    title='Overall Transaction Distribution — Fraud vs Clean'
)

fig.update_layout(
    title_font_size=16,
    title_font_color='#2D3436',
    paper_bgcolor='#F8F9FA',
    legend_title='Transaction Type',
    annotations=[dict(
        text=f'12.22%<br>CBK Rate',
        x=0.5, y=0.5,
        font_size=14,
        font_color='#2D3436',
        showarrow=False
    )]
)

fig.show()
fig.write_html('chart1_distribution.html')
print("✓ Chart 1 saved!")

✓ Chart 1 saved!


In [ ]:
# Chart 2: Chargeback Rate by Hour of Day
# Purpose: Identify high risk time windows
# Fraudsters operate at night when cardholders are asleep

hourly = df_clean.groupby('hour').agg(
    total      = ('is_fraud', 'count'),
    chargebacks= ('is_fraud', 'sum')
).reset_index()
hourly['cbk_rate'] = (hourly['chargebacks'] /
                       hourly['total'] * 100).round(2)

fig = make_subplots(specs=[[{"secondary_y": True}]])

# Bar chart — transaction volume
fig.add_trace(
    go.Bar(
        x=hourly['hour'],
        y=hourly['total'],
        name='Transaction Volume',
        marker_color='#0984E3',
        opacity=0.7
    ),
    secondary_y=False
)

# Line chart — CBK rate
fig.add_trace(
    go.Scatter(
        x=hourly['hour'],
        y=hourly['cbk_rate'],
        name='CBK Rate %',
        line=dict(color='#D63031', width=2.5),
        marker=dict(size=6)
    ),
    secondary_y=True
)

fig.update_layout(
    title='Chargeback Rate vs Volume by Hour of Day',
    title_font_size=16,
    title_font_color='#2D3436',
    paper_bgcolor='#F8F9FA',
    plot_bgcolor='#F8F9FA',
    xaxis=dict(
        title='Hour of Day',
        tickmode='linear',
        tick0=0,
        dtick=1
    ),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02
    )
)

fig.update_yaxes(
    title_text='Transaction Volume',
    secondary_y=False,
    title_font_color='#0984E3'
)
fig.update_yaxes(
    title_text='CBK Rate (%)',
    secondary_y=True,
    title_font_color='#D63031'
)

fig.show()
fig.write_html('chart2_hourly_cbk.html')
print("✓ Chart 2 saved!")

✓ Chart 2 saved!


In [ ]:
# Chart 3: Top Suspect Users by Card Count and CBK Rate
# Purpose: Identify professional fraudsters cycling through stolen cards
# These users have 81-93% chargeback rates

user_stats = df_clean.groupby('user_id').agg(
    unique_cards = ('card_number', 'nunique'),
    total_txns   = ('transaction_id', 'count'),
    chargebacks  = ('is_fraud', 'sum'),
    avg_amount   = ('transaction_amount', 'mean')
).reset_index()

user_stats['cbk_rate'] = (user_stats['chargebacks'] /
                           user_stats['total_txns'] * 100).round(1)

# Filter top 10 by unique cards
top_users = user_stats[user_stats['unique_cards'] > 1]\
            .sort_values('unique_cards', ascending=False).head(10)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Unique Cards Used per User',
        'Chargeback Rate per User'
    )
)

# Left chart — unique cards
fig.add_trace(
    go.Bar(
        x=top_users['unique_cards'],
        y=top_users['user_id'].astype(str),
        orientation='h',
        name='Unique Cards',
        marker_color='#D63031',
        text=top_users['unique_cards'],
        textposition='outside'
    ),
    row=1, col=1
)

# Right chart — CBK rate
colors = ['#D63031' if r >= 80 else
          '#FDCB6E' if r >= 50
          else '#00B894'
          for r in top_users['cbk_rate']]

fig.add_trace(
    go.Bar(
        x=top_users['cbk_rate'],
        y=top_users['user_id'].astype(str),
        orientation='h',
        name='CBK Rate %',
        marker_color=colors,
        text=top_users['cbk_rate'].astype(str) + '%',
        textposition='outside'
    ),
    row=1, col=2
)

fig.update_layout(
    title='Suspect Users — Multi-Card Profile',
    title_font_size=16,
    title_font_color='#2D3436',
    paper_bgcolor='#F8F9FA',
    plot_bgcolor='#F8F9FA',
    showlegend=False,
    height=500
)

fig.show()
fig.write_html('chart3_suspect_users.html')
print("✓ Chart 3 saved!")

✓ Chart 3 saved!


In [ ]:
# Chart 4: Risk Label Analysis
# Purpose: Show distribution of Low/Medium/High risk transactions
# and their relationship with fraud

risk_stats = df_clean.groupby('risk_label', observed=True).agg(
    total        = ('transaction_id', 'count'),
    chargebacks  = ('is_fraud', 'sum'),
    total_amount = ('transaction_amount', 'sum'),
    avg_amount   = ('transaction_amount', 'mean')
).reset_index()

risk_stats['cbk_rate'] = (risk_stats['chargebacks'] /
                           risk_stats['total'] * 100).round(1)

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        'Transaction Count',
        'CBK Rate %',
        'Average Amount (R$)'
    )
)

colors_risk = ['#00B894', '#FDCB6E', '#D63031']

# Chart 1 — transaction count
fig.add_trace(
    go.Bar(
        x=risk_stats['risk_label'],
        y=risk_stats['total'],
        marker_color=colors_risk,
        text=risk_stats['total'],
        textposition='outside',
        name='Count'
    ),
    row=1, col=1
)

# Chart 2 — CBK rate
fig.add_trace(
    go.Bar(
        x=risk_stats['risk_label'],
        y=risk_stats['cbk_rate'],
        marker_color=colors_risk,
        text=risk_stats['cbk_rate'].astype(str) + '%',
        textposition='outside',
        name='CBK Rate'
    ),
    row=1, col=2
)

# Chart 3 — average amount
fig.add_trace(
    go.Bar(
        x=risk_stats['risk_label'],
        y=risk_stats['avg_amount'].round(2),
        marker_color=colors_risk,
        text=risk_stats['avg_amount'].round(0).astype(int),
        textposition='outside',
        name='Avg Amount'
    ),
    row=1, col=3
)

fig.update_layout(
    title='Risk Label Analysis — Low / Medium / High',
    title_font_size=16,
    title_font_color='#2D3436',
    paper_bgcolor='#F8F9FA',
    plot_bgcolor='#F8F9FA',
    showlegend=False,
    height=500
)

fig.show()
fig.write_html('chart4_risk_labels.html')
print("✓ Chart 4 saved!")

✓ Chart 4 saved!


In [ ]:
# Chart 5: High Risk Merchant Analysis
# Purpose: Identify merchants with abnormally high chargeback rates
# 12 merchants had 100% CBK rate — primary fraud targets

merch = df_clean.groupby('merchant_id').agg(
    total        = ('transaction_id', 'count'),
    chargebacks  = ('is_fraud', 'sum'),
    avg_amount   = ('transaction_amount', 'mean')
).reset_index()

merch['cbk_rate'] = (merch['chargebacks'] /
                      merch['total'] * 100).round(1)

# Filter high risk merchants
top_merch = merch[
    (merch['total'] >= 5) &
    (merch['cbk_rate'] >= 30)
].sort_values('cbk_rate', ascending=True).tail(15)

# Color by CBK rate
colors = ['#D63031' if r == 100 else
          '#FDCB6E' if r >= 50
          else '#00B894'
          for r in top_merch['cbk_rate']]

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=top_merch['cbk_rate'],
        y=top_merch['merchant_id'].astype(str),
        orientation='h',
        marker_color=colors,
        text=top_merch['cbk_rate'].astype(str) + '%',
        textposition='outside',
        customdata=top_merch[['total','chargebacks','avg_amount']],
        hovertemplate=(
            'Merchant: %{y}<br>'
            'CBK Rate: %{x}%<br>'
            'Total Txns: %{customdata[0]}<br>'
            'Chargebacks: %{customdata[1]}<br>'
            'Avg Amount: R$%{customdata[2]:.2f}'
            '<extra></extra>'
        )
    )
)

fig.update_layout(
    title='High Risk Merchants — CBK Rate ≥ 30%',
    title_font_size=16,
    title_font_color='#2D3436',
    paper_bgcolor='#F8F9FA',
    plot_bgcolor='#F8F9FA',
    xaxis=dict(
        title='Chargeback Rate (%)',
        range=[0, 120]
    ),
    yaxis=dict(
        title='Merchant ID'
    ),
    height=500
)

fig.show()
fig.write_html('chart5_merchants.html')
print("✓ Chart 5 saved!")

✓ Chart 5 saved!


In [ ]:
# Chart 6: Transaction Amount Distribution
# Purpose: Compare amount distribution between fraud and clean transactions
# Fraudsters spend bigger to maximize value per stolen card

fig = go.Figure()

# Clean transactions
fig.add_trace(
    go.Histogram(
        x=df_clean[~df_clean['is_fraud']]['transaction_amount'],
        name='Clean',
        marker_color='#00B894',
        opacity=0.7,
        nbinsx=50
    )
)

# Fraud transactions
fig.add_trace(
    go.Histogram(
        x=df_clean[df_clean['is_fraud']]['transaction_amount'],
        name='Fraud',
        marker_color='#D63031',
        opacity=0.7,
        nbinsx=50
    )
)

# Add 95th percentile line
fig.add_vline(
    x=2775,
    line_dash='dash',
    line_color='#FDCB6E',
    line_width=2,
    annotation_text='95th Percentile R$2,775',
    annotation_position='top right',
    annotation_font_color='#2D3436'
)

fig.update_layout(
    title='Transaction Amount Distribution — Fraud vs Clean',
    title_font_size=16,
    title_font_color='#2D3436',
    paper_bgcolor='#F8F9FA',
    plot_bgcolor='#F8F9FA',
    barmode='overlay',
    xaxis=dict(title='Transaction Amount (R$)'),
    yaxis=dict(title='Number of Transactions'),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02
    ),
    height=450
)

fig.show()
fig.write_html('chart6_amount_dist.html')
print("✓ Chart 6 saved!")

✓ Chart 6 saved!


In [15]:
# Chart 7: Shared Cards Between Users
# Purpose: Visualize fraud ring connections
# Same card used by multiple users = impossible for legitimate use

shared = df_clean.groupby('card_number').agg(
    users        = ('user_id', 'nunique'),
    total_txns   = ('transaction_id', 'count'),
    chargebacks  = ('is_fraud', 'sum'),
    avg_amount   = ('transaction_amount', 'mean')
).reset_index()

# Filter only shared cards
shared = shared[shared['users'] > 1].copy()
shared['cbk_rate'] = (shared['chargebacks'] /
                       shared['total_txns'] * 100).round(1)

# Color by CBK rate
colors = ['#D63031' if r == 100 else
          '#FDCB6E' if r >= 50
          else '#00B894'
          for r in shared['cbk_rate']]

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=shared['card_number'],
        y=shared['cbk_rate'],
        marker_color=colors,
        text=shared['cbk_rate'].astype(str) + '%',
        textposition='outside',
        customdata=shared[['users','total_txns',
                           'chargebacks','avg_amount']],
        hovertemplate=(
            'Card: %{x}<br>'
            'Users Sharing: %{customdata[0]}<br>'
            'Total Txns: %{customdata[1]}<br>'
            'Chargebacks: %{customdata[2]}<br>'
            'CBK Rate: %{y}%<br>'
            'Avg Amount: R$%{customdata[3]:.2f}'
            '<extra></extra>'
        )
    )
)

fig.update_layout(
    title='Shared Card Numbers — Multi-User Fraud Network',
    title_font_size=16,
    title_font_color='#2D3436',
    paper_bgcolor='#F8F9FA',
    plot_bgcolor='#F8F9FA',
    xaxis=dict(
        title='Card Number',
        tickangle=45
    ),
    yaxis=dict(
        title='Chargeback Rate (%)',
        range=[0, 120]
    ),
    height=500
)

fig.show()
fig.write_html('chart7_shared_cards.html')
print("✓ Chart 7 saved!")

✓ Chart 7 saved!


In [16]:
# Chart 8: Transaction Velocity Analysis
# Purpose: Show rapid transactions by same user
# Fraudsters test stolen cards quickly before owner notices

# Calculate time difference between transactions per user
df_vel = df_clean.sort_values(['user_id', 'transaction_date'])
df_vel['prev_date'] = df_vel.groupby('user_id')['transaction_date'].shift(1)
df_vel['minutes_apart'] = (
    df_vel['transaction_date'] - df_vel['prev_date']
).dt.total_seconds() / 60

# Filter rapid transactions under 5 minutes
rapid = df_vel[
    (df_vel['minutes_apart'] > 0) &
    (df_vel['minutes_apart'] < 5)
].copy()

rapid['fraud_label'] = rapid['is_fraud'].map(
    {True: 'Fraud', False: 'Clean'})

fig = px.scatter(
    rapid,
    x='minutes_apart',
    y='transaction_amount',
    color='fraud_label',
    color_discrete_map={
        'Fraud' : '#D63031',
        'Clean' : '#00B894'
    },
    size='transaction_amount',
    hover_data=['user_id', 'card_number',
                'minutes_apart', 'transaction_amount'],
    title='Velocity Abuse — Rapid Transactions Under 5 Minutes'
)

fig.add_vline(
    x=1,
    line_dash='dash',
    line_color='#FDCB6E',
    line_width=2,
    annotation_text='1 Minute Threshold',
    annotation_font_color='#2D3436'
)

fig.update_layout(
    title_font_size=16,
    title_font_color='#2D3436',
    paper_bgcolor='#F8F9FA',
    plot_bgcolor='#F8F9FA',
    xaxis=dict(title='Minutes Apart'),
    yaxis=dict(title='Transaction Amount (R$)'),
    legend_title='Transaction Type',
    height=500
)

fig.show()
fig.write_html('chart8_velocity.html')
print("✓ Chart 8 saved!")

✓ Chart 8 saved!


In [17]:
# Chart 9: Fraud Pattern by Day of Week
# Purpose: Identify which days have highest fraud activity
# Helps plan when to increase monitoring

weekly = df_clean.groupby('weekday').agg(
    total       = ('transaction_id', 'count'),
    chargebacks = ('is_fraud', 'sum'),
    avg_amount  = ('transaction_amount', 'mean')
).reset_index()

weekly['cbk_rate'] = (weekly['chargebacks'] /
                       weekly['total'] * 100).round(1)

# Set correct day order
day_order = ['Monday','Tuesday','Wednesday',
             'Thursday','Friday','Saturday','Sunday']
weekly['weekday'] = pd.Categorical(
    weekly['weekday'],
    categories=day_order,
    ordered=True)
weekly = weekly.sort_values('weekday')

fig = make_subplots(specs=[[{"secondary_y": True}]])

# Bar chart — transaction volume
fig.add_trace(
    go.Bar(
        x=weekly['weekday'],
        y=weekly['total'],
        name='Transaction Volume',
        marker_color='#0984E3',
        opacity=0.7,
        text=weekly['total'],
        textposition='outside'
    ),
    secondary_y=False
)

# Line chart — CBK rate
fig.add_trace(
    go.Scatter(
        x=weekly['weekday'],
        y=weekly['cbk_rate'],
        name='CBK Rate %',
        line=dict(color='#D63031', width=2.5),
        marker=dict(size=8),
        text=weekly['cbk_rate'].astype(str) + '%',
        textposition='top center'
    ),
    secondary_y=True
)

fig.update_layout(
    title='Fraud Pattern by Day of Week',
    title_font_size=16,
    title_font_color='#2D3436',
    paper_bgcolor='#F8F9FA',
    plot_bgcolor='#F8F9FA',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02
    ),
    height=450
)

fig.update_yaxes(
    title_text='Transaction Volume',
    secondary_y=False,
    title_font_color='#0984E3'
)
fig.update_yaxes(
    title_text='CBK Rate (%)',
    secondary_y=True,
    title_font_color='#D63031'
)

fig.show()
fig.write_html('chart9_weekly.html')
print("✓ Chart 9 saved!")

✓ Chart 9 saved!


In [24]:
# Chart 10: Final Summary Dashboard
# Purpose: One chart showing all 5 fraud signals together
# This is your key presentation chart

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=(
        'Fraud vs Clean',
        'CBK Rate by Risk Label',
        'Avg Amount — Fraud vs Clean',
        'Missing Device Impact',
        'Night vs Day CBK Rate',
        'High Value Impact'
    ),
    specs=[
        [{"type": "pie"}, {"type": "xy"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}]
    ]
)

# --- Chart 1: Fraud vs Clean Donut ---
fig.add_trace(
    go.Pie(
        labels=['Clean', 'Fraud'],
        values=[
            df_clean['is_fraud'].value_counts()[False],
            df_clean['is_fraud'].value_counts()[True]
        ],
        hole=0.6,
        marker_colors=['#00B894', '#D63031'],
        textinfo='percent+label',
        showlegend=False
    ),
    row=1, col=1
)

# --- Chart 2: CBK Rate by Risk Label ---
risk_stats = df_clean.groupby(
    'risk_label', observed=True).agg(
    total       = ('transaction_id', 'count'),
    chargebacks = ('is_fraud', 'sum')
).reset_index()
risk_stats['cbk_rate'] = (
    risk_stats['chargebacks'] /
    risk_stats['total'] * 100).round(1)

fig.add_trace(
    go.Bar(
        x=risk_stats['risk_label'],
        y=risk_stats['cbk_rate'],
        marker_color=['#00B894', '#FDCB6E', '#D63031'],
        text=risk_stats['cbk_rate'].astype(str) + '%',
        textposition='outside',
        showlegend=False
    ),
    row=1, col=2
)

# --- Chart 3: Avg Amount Fraud vs Clean ---
avg_amounts = df_clean.groupby('is_fraud')['transaction_amount']\
              .mean().round(2).reset_index()
avg_amounts['label'] = avg_amounts['is_fraud'].map(
    {True: 'Fraud', False: 'Clean'})

fig.add_trace(
    go.Bar(
        x=avg_amounts['label'],
        y=avg_amounts['transaction_amount'],
        marker_color=['#00B894', '#D63031'],
        text=avg_amounts['transaction_amount'].astype(str),
        textposition='outside',
        showlegend=False
    ),
    row=1, col=3
)

# --- Chart 4: Missing Device Impact ---
device_stats = df_clean.groupby('missing_device').agg(
    total       = ('transaction_id', 'count'),
    chargebacks = ('is_fraud', 'sum')
).reset_index()
device_stats['cbk_rate'] = (
    device_stats['chargebacks'] /
    device_stats['total'] * 100).round(1)
device_stats['label'] = device_stats['missing_device'].map(
    {0: 'Has Device', 1: 'No Device'})

fig.add_trace(
    go.Bar(
        x=device_stats['label'],
        y=device_stats['cbk_rate'],
        marker_color=['#00B894', '#D63031'],
        text=device_stats['cbk_rate'].astype(str) + '%',
        textposition='outside',
        showlegend=False
    ),
    row=2, col=1
)

# --- Chart 5: Night vs Day CBK Rate ---
df_clean['time_window'] = df_clean['hour'].apply(
    lambda x: 'Night (21h-3h)'
    if x >= 21 or x <= 3
    else 'Day (4h-20h)')

time_stats = df_clean.groupby('time_window').agg(
    total       = ('transaction_id', 'count'),
    chargebacks = ('is_fraud', 'sum')
).reset_index()
time_stats['cbk_rate'] = (
    time_stats['chargebacks'] /
    time_stats['total'] * 100).round(1)

fig.add_trace(
    go.Bar(
        x=time_stats['time_window'],
        y=time_stats['cbk_rate'],
        marker_color=['#0984E3', '#D63031'],
        text=time_stats['cbk_rate'].astype(str) + '%',
        textposition='outside',
        showlegend=False
    ),
    row=2, col=2
)

# --- Chart 6: High Value Impact ---
df_clean['amount_tier'] = df_clean['transaction_amount'].apply(
    lambda x: 'Above R$2775'
    if x > 2775
    else 'Below R$2775')

amount_stats = df_clean.groupby('amount_tier').agg(
    total       = ('transaction_id', 'count'),
    chargebacks = ('is_fraud', 'sum')
).reset_index()
amount_stats['cbk_rate'] = (
    amount_stats['chargebacks'] /
    amount_stats['total'] * 100).round(1)

fig.add_trace(
    go.Bar(
        x=amount_stats['amount_tier'],
        y=amount_stats['cbk_rate'],
        marker_color=['#00B894', '#D63031'],
        text=amount_stats['cbk_rate'].astype(str) + '%',
        textposition='outside',
        showlegend=False
    ),
    row=2, col=3
)

fig.update_layout(
    title='Fraud Analysis — Complete Summary Dashboard',
    title_font_size=18,
    title_font_color='#2D3436',
    paper_bgcolor='#F8F9FA',
    plot_bgcolor='#F8F9FA',
    height=700,
    showlegend=False
)

fig.show()
fig.write_html('chart10_summary.html')
print("✓ Chart 10 saved!")
print("\n✓ ALL 10 CHARTS COMPLETE!")

✓ Chart 10 saved!

✓ ALL 10 CHARTS COMPLETE!


In [26]:
from google.colab import files

df_clean.to_csv('fraud_clean.csv', index=False)
files.download('fraud_clean.csv')

print(f"✓ fraud_clean.csv downloaded!")
print(f"Rows    : {len(df_clean):,}")
print(f"Columns : {df_clean.shape[1]}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ fraud_clean.csv downloaded!
Rows    : 3,199
Columns : 23
